<a href="https://colab.research.google.com/github/Kusinga321/Implementation-of-Intermediate-Algorithms-Data-Structures/blob/main/cuda_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!nvidia-smi

In [ ]:
!nvidia-smi

Thu Oct 22 21:35:09 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
%%writefile test.cu
#include <cstdio>
#include <cstdlib>
#include <cmath>

#define CHECK(call) do {                                              \
    cudaError_t err = (call);                                         \
    if (err != cudaSuccess) {                                         \
        printf("CUDA error: %s (line %d)\n",                          \
               cudaGetErrorString(err), __LINE__);                    \
        exit(1);                                                      \
    }                                                                 \
} while (0)

__global__ void add(const float* a, const float* b, float* c, int n) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) c[i] = a[i] + b[i];
}

int main() {
    // GPU info
    int dev;
    cudaDeviceProp prop;
    CHECK(cudaGetDevice(&dev));
    CHECK(cudaGetDeviceProperties(&prop, dev));
    printf("GPU: %s (compute capability %d.%d)\n",
           prop.name, prop.major, prop.minor);

    const int n = 1 << 20;  // ~1 million elements
    size_t bytes = n * sizeof(float);

    // Host memory
    float *h_a = (float*)malloc(bytes);
    float *h_b = (float*)malloc(bytes);
    float *h_c = (float*)malloc(bytes);
    for (int i = 0; i < n; i++) { h_a[i] = i; h_b[i] = 2.0f * i; }

    // Device memory
    float *d_a, *d_b, *d_c;
    CHECK(cudaMalloc(&d_a, bytes));
    CHECK(cudaMalloc(&d_b, bytes));
    CHECK(cudaMalloc(&d_c, bytes));
    CHECK(cudaMemcpy(d_a, h_a, bytes, cudaMemcpyHostToDevice));
    CHECK(cudaMemcpy(d_b, h_b, bytes, cudaMemcpyHostToDevice));

    // Launch kernel
    int threads = 256;
    int blocks = (n + threads - 1) / threads;
    add<<<blocks, threads>>>(d_a, d_b, d_c, n);
    CHECK(cudaGetLastError());
    CHECK(cudaDeviceSynchronize());

    // Copy back and verify
    CHECK(cudaMemcpy(h_c, d_c, bytes, cudaMemcpyDeviceToHost));
    int errors = 0;
    for (int i = 0; i < n; i++) {
        if (fabsf(h_c[i] - 3.0f * i) > 1e-3f * fmaxf(1.0f, 3.0f * i)) errors++;
    }

    if (errors == 0) printf("SUCCESS: all %d results are correct!\n", n);
    else             printf("FAILED: %d wrong results\n", errors);

    cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);
    free(h_a); free(h_b); free(h_c);
    return errors ? 1 : 0;
}

Writing test.cu


In [ ]:
!nvcc -arch=sm_75 test.cu -o test && ./test

GPU: Tesla T4 (compute capability 7.5)
SUCCESS: all 1048576 results are correct!
